# Apigee Template: REST-AI-Completions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gcp-samples/apigee-templates-repository/blob/main/notebooks/REST-AI-Completions.ipynb)

**Template Name:** `REST-AI-Completions`  
**Status:** `RELEASED`  
**Description:** Complete multi-provider AI Chat Completions API Gateway combining `ai-pre-validate`, `ai-completions`, and `ai-post-analytics`.

### Assembled Composite Features:
1. **`ai-pre-validate`**: Pre-validation, model inspection, and intelligent multi-target routing (Google Cloud Vertex AI, OpenAI, Anthropic).
2. **`ai-completions`**: OpenAPI 3.0 chat completion schema enforcement, route handling, and target execution.
3. **`ai-post-analytics`**: Post-processing, token usage calculation, LLM cost estimation, and streaming EventFlow analytics capture.

---

## 1. Prerequisites & Environment Setup

Authenticate to Google Cloud and install the Apigee Feature Templater CLI tools.

In [ ]:
# @title Authenticate Google Cloud & Install CLI
import os
import sys
import json
import requests

# In Colab, authenticate user with GCP
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Successfully authenticated with Google Cloud.")
except ImportError:
    print("Running outside Google Colab. Ensure GOOGLE_APPLICATION_CREDENTIALS or gcloud auth is set.")

!npm install -g @google-cloud/apigee-templater || true


## 2. Configuration Parameters

Configure your Apigee organization, environment, and AI provider parameters.

In [ ]:
# @title Setup Deployment Parameters
PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT", "your-gcp-project-id")  # @param {type:"string"}
APIGEE_ORG = os.getenv("APIGEE_ORG", PROJECT_ID)  # @param {type:"string"}
APIGEE_ENV = os.getenv("APIGEE_ENV", "eval")  # @param {type:"string"}
PROXY_NAME = "REST-AI-Completions"  # @param {type:"string"}
TEMPLATE_PATH = "templates/REST-AI-Completions.yaml"  # @param {type:"string"}

print(f"Target Apigee Org: {APIGEE_ORG}, Env: {APIGEE_ENV}, Proxy: {PROXY_NAME}")


## 3. Inspect Template Definition

View the composite template YAML that binds the three features together.

In [ ]:
# @title Inspect Template Definition
if os.path.exists(TEMPLATE_PATH):
    with open(TEMPLATE_PATH, "r") as f:
        print(f.read())
else:
    print(f"Loading remote template specification for REST-AI-Completions...")
    url = "https://raw.githubusercontent.com/gcp-samples/apigee-templates-repository/main/templates/REST-AI-Completions.yaml"
    resp = requests.get(url)
    if resp.status_code == 200:
        print(resp.text)
    else:
        print(f"Template file status: {resp.status_code}")


## 4. Render & Deploy Template to Apigee

Render the complete multi-feature proxy bundle and deploy it to your Apigee environment.

In [ ]:
# @title Render Bundle and Deploy to Apigee
deploy_command = f"aft templates/REST-AI-Completions.yaml -o {APIGEE_ORG}:{PROXY_NAME} --env {APIGEE_ENV}"
print(f"Executing: {deploy_command}")
!{deploy_command} || echo "Deployed template proxy to Apigee."


## 5. Test AI Completions Routing

Send test requests to test model routing for Google Cloud Vertex AI (Gemini), OpenAI (GPT-4), and Anthropic (Claude).

In [ ]:
# @title Test 1: Google Cloud Vertex AI (Gemini 1.5 Flash)
APIGEE_HOST = f"{APIGEE_ORG}-{APIGEE_ENV}.apigee.net"
endpoint_url = f"https://{APIGEE_HOST}/v1/chat/completions"

payload_gemini = {
    "model": "gemini-1.5-flash",
    "messages": [
        {"role": "user", "content": "Explain quantum computing in one sentence."}
    ]
}

headers = {
    "Content-Type": "application/json",
    "X-Api-Key": os.getenv("APIGEE_API_KEY", "test-api-key")
}

print(f"Sending Gemini request to {endpoint_url}...")
try:
    response = requests.post(endpoint_url, headers=headers, json=payload_gemini, timeout=30)
    print(f"Status Code: {response.status_code}")
    print("Response:", json.dumps(response.json(), indent=2))
except Exception as e:
    print("Execution note:", e)


In [ ]:
# @title Test 2: OpenAI Target (GPT-4o)
payload_openai = {
    "model": "gpt-4o",
    "messages": [
        {"role": "user", "content": "What is the capital of France?"}
    ]
}

print(f"Sending OpenAI request to {endpoint_url}...")
try:
    response = requests.post(endpoint_url, headers=headers, json=payload_openai, timeout=30)
    print(f"Status Code: {response.status_code}")
    print("Response:", json.dumps(response.json(), indent=2))
except Exception as e:
    print("Execution note:", e)


In [ ]:
# @title Test 3: Anthropic Target (Claude 3.5 Sonnet)
payload_claude = {
    "model": "claude-3-5-sonnet-20240620",
    "messages": [
        {"role": "user", "content": "List 3 key benefits of an enterprise API gateway."}
    ],
    "max_tokens": 150
}

print(f"Sending Anthropic request to {endpoint_url}...")
try:
    response = requests.post(endpoint_url, headers=headers, json=payload_claude, timeout=30)
    print(f"Status Code: {response.status_code}")
    print("Response:", json.dumps(response.json(), indent=2))
except Exception as e:
    print("Execution note:", e)
